# Hyperparameter Tuning

Tune learning rate, batch size, and optimizer for optimal performance.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from sklearn.model_selection import ParameterGrid
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load data
checkpoint = torch.load('../data/processed_datasets.pth', map_location='cpu')
train_dataset = checkpoint['train_dataset']
val_dataset = checkpoint['val_dataset']

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4)

In [ ]:
# Define hyperparameter grid
param_grid = {
    'learning_rate': [0.01, 0.001, 0.0001],
    'batch_size': [16, 32, 64],
    'optimizer': ['adam', 'sgd']
}

# Create parameter combinations
param_combinations = list(ParameterGrid(param_grid))
print(f"Total combinations: {len(param_combinations)}")

In [ ]:
# Quick training function for hyperparameter search
def quick_train(params, num_epochs=3):
    # Create model
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
    for param in model.parameters():
        param.requires_grad = False
    model.fc = nn.Linear(2048, 102)
    model = model.to(device)
    
    # Setup optimizer
    if params['optimizer'] == 'adam':
        optimizer = optim.Adam(model.fc.parameters(), lr=params['learning_rate'])
    else:
        optimizer = optim.SGD(model.fc.parameters(), lr=params['learning_rate'], momentum=0.9)
    
    criterion = nn.CrossEntropyLoss()
    
    # Create data loader with specific batch size
    temp_train_loader = torch.utils.data.DataLoader(
        train_dataset, batch_size=params['batch_size'], shuffle=True, num_workers=4
    )
    
    # Train for a few epochs
    best_val_acc = 0
    for epoch in range(num_epochs):
        model.train()
        for inputs, labels in temp_train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
        
        # Validate
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)
        val_acc = correct / total
        best_val_acc = max(best_val_acc, val_acc)
    
    return best_val_acc

In [ ]:
# Run hyperparameter search
results = []
for i, params in enumerate(param_combinations):
    print(f"Testing combination {i+1}/{len(param_combinations)}: {params}")
    val_acc = quick_train(params)
    results.append({**params, 'val_acc': val_acc})
    print(f"  Validation accuracy: {val_acc:.4f}")

In [ ]:
# Find best hyperparameters
best_result = max(results, key=lambda x: x['val_acc'])
print("\nBest hyperparameters:")
for key, value in best_result.items():
    print(f"  {key}: {value}")